[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/06-attention.ipynb)

# Module 6: Attention

Companion notebook to [Module 6](https://transformers-and-math.drmhse.com/modules/06-attention) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Part 1 — Scaled dot-product attention from scratch (section 6.2)
The two-token example from the module, step by step.

In [ ]:
Q = torch.tensor([[3., -3.], [1., 1.]])
K = torch.tensor([[-3., 3.], [1., 1.]])
V = torch.tensor([[0., -1.], [1., 2.]])

def attention(Q, K, V, mask=None):
    scores = Q @ K.T / math.sqrt(K.shape[-1])
    if mask is not None:
        scores = scores + mask
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights

causal = torch.triu(torch.full((2, 2), float('-inf')), diagonal=1)
out, w = attention(Q, K, V, causal)
print('weights:\n', w.round(decimals=3), '\noutput:\n', out.round(decimals=3))
print('matches PyTorch:', torch.allclose(out, nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=True)))

## Part 2 — Why divide by √d_k (section 6.3)
Unscaled scores spread like √d_k and saturate the softmax.

In [ ]:
for dk in (16, 64, 128, 256):
    q, k = torch.randn(500, 1, dk), torch.randn(500, 64, dk)
    raw = (q * k).sum(-1)
    for scaled in (False, True):
        s = raw / math.sqrt(dk) if scaled else raw
        pmax = torch.softmax(s, -1).max(-1).values
        print(f'd_k={dk:4}  scaled={scaled!s:5}  score sd={s.std():6.2f}  share with p_max>0.99: {(pmax > 0.99).float().mean():.0%}')

## Part 3 — Masks (section 6.4)

In [ ]:
N = 5
causal = torch.triu(torch.full((N, N), float('-inf')), diagonal=1)
print(causal)
lengths = torch.tensor([5, 3])                      # second sequence padded after 3 tokens
padding = torch.where(torch.arange(N)[None, :] < lengths[:, None], 0.0, float('-inf'))
print('padding mask for the short sequence:', padding[1])

## Part 4 — Attention cannot see word order (section 6.5)

In [ ]:
X = torch.randn(4, 8)
Wq, Wk, Wv = torch.randn(8, 8), torch.randn(8, 8), torch.randn(8, 8)
self_attn = lambda X: attention(X @ Wq, X @ Wk, X @ Wv)[0]
perm = torch.tensor([2, 0, 3, 1])
print('shuffle in = shuffle out:', torch.allclose(self_attn(X[perm]), self_attn(X)[perm], atol=1e-5))

## Part 5 — Multi-head attention and its cost (section 6.6)

In [ ]:
class MultiHead(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        self.h, self.dk = h, d_model // h
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
    def forward(self, X, mask=None):
        N = X.shape[0]
        q, k, v = self.qkv(X).view(N, 3, self.h, self.dk).unbind(1)
        heads = [attention(q[:, i], k[:, i], v[:, i], mask)[0] for i in range(self.h)]
        return self.out(torch.cat(heads, dim=-1))

mha = MultiHead(512, 8)
print('output shape:', tuple(mha(torch.randn(10, 512)).shape))
d = 4096
print('the N² term overtakes the N·d² term at N >', 2 * d)

## Your turn
1. Compute one attention output by hand for scores $[2, 0, 0]$ and scalar values $[1, 4, 7]$, then check it in code.
2. Add a position vector to each row of `X` in Part 4 (for example `torch.randn(4, 8)` added by slot) and show the equivariance breaks.

<details><summary>Answer</summary>

1. About 1.959: `torch.softmax(torch.tensor([2., 0., 0.]), 0) @ torch.tensor([1., 4., 7.])`.
</details>